In [1]:
import sys

# If a previous `import torch` failed mid-flight, Python can keep a *broken*
# `torch` in sys.modules — the next import returns that stub and crashes with:
# NameError: name '_C' is not defined
# Clearing torch* from sys.modules forces a clean re-import (or use Kernel → Restart).
for _k in list(sys.modules.keys()):
    if _k == "torch" or _k.startswith("torch."):
        del sys.modules[_k]

# Wrong kernel → PyTorch init may fail similarly. Fix kernel or:
# uv run python -m ipykernel install --user --name 3dgcl --display-name "3dgcl (uv)"
if ".venv" not in sys.executable.replace("/", "\\"):
    raise RuntimeError(
        "커널이 프로젝트 .venv 가 아닙니다. 현재 실행 파일:\n"
        + sys.executable
    )

# Windows CPU venv + conda base: preload torch DLL dir before native code loads (DLL search order mixups).
import os

if sys.platform == "win32":
    try:
        from pathlib import Path

        _venv = Path(sys.executable).resolve().parent.parent
        _torch_lib = _venv / "Lib" / "site-packages" / "torch" / "lib"
        if _torch_lib.is_dir():
            os.add_dll_directory(str(_torch_lib))
            _pb = sys.base_exec_prefix
            _conda_bin = os.path.join(_pb, "Library", "bin") if _pb != sys.prefix else ""
            if _conda_bin and os.path.isdir(_conda_bin):
                os.add_dll_directory(_conda_bin)
    except OSError:
        pass

import torch

print(torch.__version__)
print("cuda available:", torch.cuda.is_available())


2.2.2+cpu
cuda available: False


In [2]:
from IPython.display import display, HTML

display(HTML("<style>.container { width:90% !important; }</style>"))

import sys

sys.path.insert(0, "..")
sys.path.insert(0, "../..")

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

# `import torch` only in cell 0 — re-import here can resurrect a stale partial torch module.
from torch.utils.tensorboard import SummaryWriter

from dig.sslgraph.utils import Encoder
from dig.sslgraph.utils.device import pick_torch_device
from dig.sslgraph.evaluation import Pretrain
from dig.threedgraph.dataset import MoleculeNet, QM
from dig.sslgraph.method import GraphCL

print("pick_torch_device() ->", pick_torch_device())



In [ ]:
# 위에서 torch(셀0)와 dig import(셀1)까지 실행한 뒤 실행하세요.

def main():
    import argparse as _argparse

    _parser = _argparse.ArgumentParser()
    args = _parser.parse_args([])

    # Finetune or rand init
    args.finetune = False
    args.seed = 2222

    # File Path
    args.model_path = './models'

    # Device — CUDA → NPU(torch.npu) → MPS → Intel XPU → (optional DirectML) → CPU
    args.device = pick_torch_device()

    # Dataset
    args.pretrain_dataset = 'esol'
    args.batch_size = 400
    
    # Model
    args.encoder = 'schnet'
    args.edge_weight = True
    args.feat_dim = 9
    args.cutoff = 5.0  # [5.0, 10.0]
    args.num_layers = 2 # [2, 4]
    args.num_filters = 128
    args.num_gaussians = 50
    args.z_dim = 32
    
    args.int_emb_size = 64
    args.basis_emb_size_dist = 8
    args.basis_emb_size_angle = 8
    args.basis_emb_size_torsion = 8
    args.out_emb_channels = 256
    args.num_spherical = 3
    args.num_radial = 6
    args.envelope_exponent = 5
    args.num_before_skip = 1
    args.num_after_skip = 2
    args.num_output_layers = 3
    args.use_node_features = True

    # Learning
    args.p_epoch = 100
    args.p_lr = 1e-3
    args.aug_1, args.aug_2 = 'MMFFrandom', 'MMFFrandom'
    args.aug_ratio = 0.25
    args.tau = 0.2
    args.proj = 'spherenet'

    # Regularization
    args.dropout_rate = 0.0

    args.p_optim = 'ExponentialLR' #['StepLR', ExponentialLR, 'Cosine']
    
    #'StepLR'
    args.p_weight_decay = 0
    args.p_lr_decay_step_size = 15  # 15 epoch 마다 lr * p_lr_decay_factor
    args.p_lr_decay_factor = 0.5

    # ExponentialLR
    args.expo_gamma = 0.95
    
    # Cosine
    args.T_0 = 20        # 최초 주기값
    args.T_mult = 2      # 최초 주기값에 비해 얼만큼 주기를 늘려갈 것인지
    args.eta_max = 0.05  # lr 최대값
    args.T_up = 10      # Warm up 시 필요한 epoch 수(일반적으로 짧은 수)
    args.gamma = 0.5     # 주기가 반복될수록 곱해지는 scale 값

    args.pc = False
    
    encoder = Encoder(args)
    graphcl = GraphCL(args)  # , device=args.device
    evaluator = Pretrain(args)
    encoder = evaluator.evaluate(learning_model=graphcl, encoder=encoder)


main()

Pretraining: epoch 1:   0%|          | 0/100 [00:00<?, ?it/s]

./models/encoder-schnet_pretrain-esol_batch-400_proj-spherenet_cutoff-5.0_layers-2_filter-128_gau-50_z_dim-32_lr-0.001_aug_1-MMFFrandom_aug_2-MMFFrandom_aug_ratio-0.25_tau-0.2_optim-ExponentialLR_weight_decay-0_expo_gamma-0.95_dropout-0.0


Pretraining: epoch 17:  16%|█▌        | 16/100 [02:00<11:54,  8.50s/it, loss=9.2012]